# 5-5: Word Embeddings in Python

In this tutorial, we'll learn how to work with __word embeddings__. A word embedding represents a word as a vector: a list of numbers learned from word context. Basically, as the model reads through a corpus, it repeatedly asks which words tend to appear near one another, then adjusts the numbers in each vector so words used in similar contexts end up closer together in vector space.

This idea comes from the __distributional hypothesis__: words that appear in similar contexts tend to have related meanings. For example, if `mother` and `father` often appear near similar surrounding words, an embedding model may place their vectors near each other. If `money` often appears near words about inheritance, payment, debt, or income, those contexts shape its vector. A famous shorthand example is `king - man + woman ~= queen`. The point is not that the model understands monarchy the way a human does. The model just notes how repeated context patterns can encode relationships among words, including relationships like gender, social roles, genre, topic, and historical usage.

Word embeddings are therefore not like dictionary definitions. They are measurements of usage patterns in a corpus. For DH work, this makes them useful but also interpretive: an embedding can suggest how words are patterned in a collection of texts, but we still need to connect those patterns back to corpus construction, historical context, and close reading.

Word embeddings, it should be noted, are also a core component of LLMs. Before an LLM can process text, it has to convert tokens into numeric vectors. Those vectors give the model a mathematical way to represent relationships among words and word pieces. Modern LLMs use much more complex embeddings than the small Word2Vec models we'll train here, but the basic idea is related: language is transformed into numbers so the model can compare, combine, and predict patterns in text.

We'll train our own small embedding models with `gensim`'s Word2Vec, using two nineteenth-century fiction corpora we've seen before:

1. Jane Austen novels
2. Charles Dickens novels

Then we'll compare those corpus-specific embeddings with pretrained word vectors from `spaCy`. By the end, you should be able to explain how Word2Vec learns from word contexts, why embeddings are useful for machine learning, and how to interpret embedding results carefully in a DH project.

## Learning Objectives

1. Explain what a word embedding is
2. Explain how Word2Vec learns vectors from word contexts
3. Describe how embeddings relate to TF-IDF, classification, topic modeling, and NER
4. Use spaCy to tokenize text for embedding training
5. Train Word2Vec models with `gensim`
6. Compare nearest neighbors and cosine similarities across corpora
7. Visualize selected word vectors with PCA
8. Use spaCy pretrained vectors as a point of comparison
9. Interpret embedding output cautiously as corpus-specific evidence


## Word Embeddings As Corpus Evidence

Because embeddings measure patterns of use, we should always treat them as evidence about a corpus rather than as final statements about meaning. Remember: this is distributional semantics in practice. A word's meaning always comes from the context in which it is used. So, when we ask which words are closest to `money`, `marriage`, or `work`, we are asking which words occupy similar contextual positions in these particular texts. The results can point us toward meaningful patterns, but they should always be checked against examples from the novels themselves.

## How Embeddings Relate To Other Machine-Learning Methods

We've already seen several ways of turning text into computational evidence:

| method | what it does |
| --- | --- |
| Word counts | counts how often words appear |
| TF-IDF | weights words by frequency and distinctiveness in texts |
| Topic modeling | finds clusters of words that tend to appear together |
| Classification | learns patterns between text features and labels |
| NER | predicts spans and labels for named entities |
| Word embeddings | learns numeric vectors from word contexts |

Like TFIDF or word frequency, word embeddings often become __features__ for other machine-learning methods or computational pipelines. For example, a classifier can use embeddings instead of TF-IDF features. A search system can use embeddings to find texts that are semantically similar rather than only texts that share exact keywords. A diachronic analysis of texts can use embeddings to see which words retain or change meaning over time.

Word2Vec, a machine-learning model in the `Gensim` library, learns from data. It doesn't use a hand-coded dictionary of meanings. It trains a small neural model to predict words from nearby words, or nearby words from a target word. In doing so, it learns vectors that encode context patterns.

This kind of learning is often called __self-supervised__ learning. We don't need human labels like `positive` or `negative`. The model creates a training task from the text itself: use nearby words as evidence.

## The Corpora

We'll use two local CSV files:

- `../data/jane_austen_texts.csv`
- `../data/dickens_texts.csv`

Remember these from our previous tutorials? Each row is a novel. Each file includes the title, publication year, author, and full text.

This setup lets us ask corpus-comparison questions. For example:

- Which words appear near `money` in Austen and Dickens?
- Does `marriage` have similar neighbors in both corpora?
- Which words are close to `work`, `home`, `child`, or `letter`?

Word embeddings are useful for such questions because they allow us to study the semantic differences between the same words in different corpora. This means we can get at the differences between the meanings of things like money, marriage, or home as they're represented in Austen's versus Dickens's works.

## Setup

All of the libraries below should be familiar to you by now, but notice two new imports: `Word2Vec` and `PCA`. `Word2Vec` is the `gensim` model we'll use to train word embeddings from our Austen and Dickens corpora. `PCA` comes from `scikit-learn`. We'll use it to compress high-dimensional word vectors into two dimensions so we can plot them.

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import spacy
from spacy.cli import download

from gensim.models import Word2Vec
from sklearn.decomposition import PCA

We'll start with spaCy's small English model for tokenization. Like the other __tokenizers__ we've used, it breaks strings into smaller units (tokens): words, punctuation marks, contractions, numbers, and other meaningful pieces of text. We've already used Python's `.split()` method, which separates text wherever it sees spaces. That's fast and easy, but it's also a blunt instrument: it doesn't consider punctuation, contractions, or English-language conventions. We've also used NLTK's tokenizer, which is more linguistically informed than `.split()`. spaCy's tokenizer is comparable to NLTK's tokenizer. It's another linguistically informed option. It uses rules for English tokenization, so it can handle many common cases more carefully than splitting on spaces. We're using spaCy here because Word2Vec needs many tokenized sequences, and spaCy can process long corpora efficiently.

The object we create below, `preprocess_nlp`, is a spaCy language pipeline. When we pass text into it, it returns a spaCy `Doc` object: an ordered sequence of token objects. Each token object has useful attributes, such as `token.text`, `token.is_alpha`, and `token.is_stop`, that we'll use during preprocessing. We disable heavier components like the parser and named entity recognizer because, for this part of the tutorial, we only need tokenization and basic token attributes. Disabling those components makes preprocessing faster.

In [ ]:
def load_spacy_model(model_name, disable=None):
    if disable is None:
        disable = []

    # notice the disabled components
    try:
        return spacy.load(model_name, disable=disable)
    except OSError:
        download(model_name)
        return spacy.load(model_name, disable=disable)


preprocess_nlp = load_spacy_model(
    "en_core_web_sm",
    disable=["tagger", "parser", "ner", "attribute_ruler", "lemmatizer"],
)

preprocess_nlp.pipe_names

## Step 1: Load The Corpora

Let's read the Austen and Dickens CSV files from our course data folder:

In [ ]:
austen_texts = pd.read_csv("../data/jane_austen_texts.csv")
dickens_texts = pd.read_csv("../data/dickens_texts.csv")

print(austen_texts.shape)
print(dickens_texts.shape)


Let's preview the data. Remember what's in these corpora?

In [ ]:
austen_texts.head()

In [ ]:
dickens_texts.head()

We can also combine the two tables for a simple corpus summary. The next cell adds a `corpus` label to each table so we can keep Austen and Dickens separate after combining them. Then it joins the two tables into one dataframe, calculates the number of characters in each full text, and displays a compact list of titles, publication years, corpus labels, and text lengths.

In [ ]:
austen_texts["corpus"] = "Austen"
dickens_texts["corpus"] = "Dickens"

all_texts = pd.concat([austen_texts, dickens_texts], ignore_index=True)
all_texts["character_count"] = all_texts["full_text"].str.len()

all_texts[["corpus", "title", "year_published", "character_count"]]

The next cell summarizes the combined dataframe by corpus. It groups the rows by `corpus`, then calculates three basic measures: how many texts are in each corpus, the total number of characters across those texts, and the mean number of characters per text. This gives us a quick sense of the relative size of the Austen and Dickens materials before we train models on them.

In [ ]:
(
    all_texts
    .groupby("corpus")
    .agg(
        texts=("title", "count"),
        total_characters=("character_count", "sum"),
        mean_characters=("character_count", "mean"),
    )
)

## Step 2: Preprocess Text For Word2Vec

Word2Vec expects a list of token lists because it learns from local word context. The outer list is the whole training corpus, and each inner list is one sequence of tokens where nearby words can be treated as context for one another. Each inner list is often called a sentence, but it doesn't have to be a perfect grammatical sentence. What matters is that words appearing near each other in the list count as context.

We'll create short text chunks and tokenize those chunks with spaCy. Our preprocessing will:

1. lowercase words
2. keep alphabetic tokens
3. remove stop words
4. remove very short tokens

These are choices. Keeping stop words or lemmatizing words would produce different embeddings. As always, remember to carefully consider your preprocessing steps when doing any textual analysis!

To do our preprocessing, we'll break it into two functions. The first is `make_text_chunks()`. This function takes one long text, splits it into paragraphs, removes empty paragraphs, and then bundles those paragraphs into chunks of about 2,500 characters. Chunking keeps us from sending an entire novel through spaCy as one enormous document. This would provide far more textual context, but it's slower and overkill. We don't need a whole novel's worth of text to get the semantic relationships of words. The function also gives Word2Vec shorter training lines where words are still close enough to count as meaningful context.

In [ ]:
def make_text_chunks(text, max_chars=2500):

    paragraphs = []
    for paragraph in text.split("\n\n"):
        if paragraph.strip():
            paragraphs.append(paragraph.strip())

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        if len(current_chunk) + len(paragraph) + 1 <= max_chars:
            current_chunk = (current_chunk + " " + paragraph).strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = paragraph[:max_chars]

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

Let's call our second function `preprocess_corpus()`. This function takes a collection of full texts and the spaCy pipeline we loaded above. First, it loops through every text and uses `make_text_chunks()` to turn the novels into shorter chunks. Then it sends those chunks through `nlp.pipe()`, which is spaCy's efficient way of processing many texts in batches.

For each spaCy `Doc`, the function builds a cleaned token list. It lowercases each token, keeps only alphabetic tokens, removes spaCy stop words, and drops tokens with two or fewer characters. Finally, it keeps only token lists that have at least `min_tokens` tokens. The returned value is exactly what Word2Vec needs: a list where each item is a cleaned sequence of tokens.

In [ ]:
def preprocess_corpus(texts, nlp, min_tokens=5):
    chunks = []

    for text in texts:
        chunks.extend(make_text_chunks(text))

    token_lists = []

    for doc in nlp.pipe(chunks, batch_size=100):
        tokens = []

        for token in doc:
            token_text = token.text.lower()

            if token.is_alpha and not token.is_stop and len(token_text) > 2:
                tokens.append(token_text)

        if len(tokens) >= min_tokens:
            token_lists.append(tokens)

    return token_lists

Let's test the preprocessing on one small passage before applying it to the full corpora. How does it work? What does it create?

In [ ]:
sample_passage = austen_texts.loc[1, "full_text"][:1000]
sample_tokens = preprocess_corpus([sample_passage], preprocess_nlp)

sample_tokens[0][:50]


Now let's preprocess the full Austen and Dickens corpora. This may take a minute or two because spaCy is tokenizing several long novels. Once done, though, we can compare the two authors' collections of works by `training lines` and `tokens`. The training lines are the cleaned token lists that Word2Vec will read as context windows. In our preprocessing, each training line comes from one text chunk. The tokens are the individual word units inside those training lines after lowercasing, stop-word removal, and filtering.

In [ ]:
austen_sentences = preprocess_corpus(austen_texts["full_text"], preprocess_nlp)
dickens_sentences = preprocess_corpus(dickens_texts["full_text"], preprocess_nlp)

print(f"Austen training lines: {len(austen_sentences)}")
print(f"Austen tokens: {sum(len(sentence) for sentence in austen_sentences):,}")
print(f"Dickens training lines: {len(dickens_sentences)}")
print(f"Dickens tokens: {sum(len(sentence) for sentence in dickens_sentences):,}")

Let's inspect one training line. This is the kind of input Word2Vec will use to learn context patterns:

In [ ]:
austen_sentences[10][:40]

In [ ]:
austen_sentences[0]

## Step 3: Train Word2Vec Models

Now we'll train two Word2Vec models: one on Austen and one on Dickens. Word2Vec has several parameters that shape what the model learns. These settings control the size of each vector, how much surrounding context the model considers, which words are frequent enough to include, which training algorithm it uses, and how many times it trains over the corpus. Changing these settings can change the resulting neighborhoods and similarity scores, so they should be treated as interpretive choices rather than neutral defaults.

A few important settings:

- `vector_size=100`: each word becomes a vector with 100 numbers. A larger vector can represent more complexity, but it also needs more data to train reliably.
- `window=5`: the model looks up to five words to the left and right of a target word. A smaller window tends to emphasize tighter syntactic or phrase-level relationships; a larger window tends to capture broader topical relationships.
- `min_count=5`: ignore words that appear fewer than five times. This reduces noise from rare words, OCR artifacts, and one-off names, but it also means some interesting rare terms will be left out.
- `sg=1`: use skip-gram training. Skip-gram tries to predict nearby context words from a target word and often works reasonably well for smaller corpora.
- `epochs=15`: train over the corpus fifteen times. More epochs give the model more chances to adjust the vectors, though too much training can overfit a small corpus.
- `seed=42` and `workers=1`: make the results more reproducible. Word2Vec includes random initialization, so these settings help us get the same output when we rerun the notebook.

For more on these settings and other options, [see documentation here](https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec).

In [ ]:
w2v_settings = {
    "vector_size": 100,
    "window": 5,
    "min_count": 5,
    "workers": 1,
    "seed": 42,
    "sg": 1,
    "epochs": 15,
}

austen_model = Word2Vec(sentences=austen_sentences, **w2v_settings)
dickens_model = Word2Vec(sentences=dickens_sentences, **w2v_settings)

print("Austen vocabulary size:", len(austen_model.wv))
print("Dickens vocabulary size:", len(dickens_model.wv))

The `.wv` part of the model stores the learned word vectors. Let's look at one vector:

In [ ]:
austen_model.wv["money"][:10]

The code above shows the first ten numbers in Austen's vector for `money`. On their own, those numbers don't tell us much. We should not read the first number as one specific idea, the second number as another idea, and so on. The vector only becomes meaningful when we compare it with other word vectors from the same model.

One common comparison is __cosine similarity__. Cosine similarity asks whether two vectors point in a similar direction. If two words tend to appear in similar contexts, their vectors will usually point in more similar directions, and their cosine similarity score will be higher. If two words appear in very different contexts, their vectors will point in less similar directions, and their score will be lower.

In the next code cell, we'll compare `money` and `marriage` in the Austen model and in the Dickens model. The scores are not percentages, and they are not direct statements about what the words "really" mean. They are model-based measurements of contextual similarity. So, if Austen's `money` / `marriage` score is higher than Dickens's, that suggests that those two words occupy more similar contexts in the Austen model than they do in the Dickens model. That is a clue for interpretation, not a conclusion by itself.

Let's compare some words and their similarities in these novels. What do their similarities tell us about the meanings of these words within each author's works?

In [ ]:
print("Austen money/marriage similarity:", austen_model.wv.similarity("money", "marriage"))
print("Dickens money/marriage similarity:", dickens_model.wv.similarity("money", "marriage"))

## Step 4: Compare Nearest Neighbors Across Corpora

A common way to interpret word embeddings is to ask for a word's nearest neighbors. These are the words with vectors closest to the query word.

Remember the caution: the Austen and Dickens models were trained separately. We are not comparing raw coordinates. We are comparing the neighborhood around the same query word inside each model.

The function below gives us a reusable way to inspect a word's nearest neighbors in one model. It takes a trained Word2Vec model, a `query_word`, a label for the corpus, and the number of neighbors we want to return. First, it checks whether the query word is actually in the model's vocabulary. If the word was filtered out by `min_count` or never appeared in the corpus, the function returns a one-row dataframe saying `NOT IN VOCABULARY`.

If the word is available, the function calls `model.wv.most_similar()`. This asks gensim to find the words whose vectors have the highest cosine similarity to the query word. The function stores each neighbor and similarity score in a list of dictionaries, then returns the results as a pandas dataframe so they are easy to read, sort, combine, and plot later.

In [ ]:
def nearest_neighbors(model, query_word, corpus_name, topn=10):
    if query_word not in model.wv:
        return pd.DataFrame({
            "corpus": [corpus_name],
            "query": [query_word],
            "neighbor": ["NOT IN VOCABULARY"],
            "similarity": [np.nan],
        })

    rows = []

    for neighbor, similarity in model.wv.most_similar(query_word, topn=topn):
        rows.append({
            "corpus": corpus_name,
            "query": query_word,
            "neighbor": neighbor,
            "similarity": similarity,
        })

    return pd.DataFrame(rows)

The next function compares the same query word across our two author-specific models. Notice how it uses `nearest_neighbors()` once for the Austen model and once for the Dickens model, then combines the two result tables with `pd.concat()`. This lets us read the neighborhoods side by side while still remembering that each neighborhood comes from its own separately trained model.

In [ ]:
def compare_neighbors(query_word, topn=10):
    return pd.concat([
        nearest_neighbors(austen_model, query_word, "Austen", topn=topn),
        nearest_neighbors(dickens_model, query_word, "Dickens", topn=topn),
    ], ignore_index=True)

Let's start with `money`.


In [ ]:
compare_neighbors("money", topn=10)

Now try `marriage`.


In [ ]:
compare_neighbors("marriage", topn=10)

And `work`.


In [ ]:
compare_neighbors("work", topn=10)

### Challenge: Interpret A Neighborhood

Choose one query word from the list below and compare its nearest neighbors in Austen and Dickens.

Possible query words:

- `home`
- `children`
- `women`
- `letter`
- `father`
- `mother`
- `london`
- `business`
- `death`
- `hope`

Ask:

1. Which neighbors make intuitive sense?
2. Which neighbors surprise you?
3. Do the two corpora suggest different contexts for the word?
4. Which result would you want to investigate by returning to the novels?

In [ ]:
compare_neighbors("children", topn=5)

## Step 5: Compare Word-Pair Similarities

Nearest neighbors are useful, but we can also compare specific word pairs.

For each pair below, we'll calculate cosine similarity in the Austen model and the Dickens model.

Again, these numbers are exploratory. Because the models were trained separately, we should treat cross-corpus comparison as a prompt for interpretation and comparative analysis. We're asking: what do these words mean in different authors' works?

The function below builds a table of cosine similarities for specific word pairs. It expects `pairs` to be a list of two-word tuples, such as `("money", "marriage")`. For each pair, it checks the Austen model and the Dickens model. If both words are in a model's vocabulary, it calculates their cosine similarity with `model.wv.similarity()`. If either word is missing, it records `np.nan`, which pandas uses for missing numeric values.

Each result is stored as one row with four pieces of information: `word_1`, `word_2`, `corpus`, and `similarity`. Returning the results as a dataframe makes it easier to compare the same relationship across corpora.

In [ ]:
def word_pair_similarity_table(pairs):
    rows = []

    for word_1, word_2 in pairs:
        for corpus_name, model in [("Austen", austen_model), ("Dickens", dickens_model)]:
            if word_1 in model.wv and word_2 in model.wv:
                similarity = model.wv.similarity(word_1, word_2)
            else:
                similarity = np.nan

            rows.append({
                "word_1": word_1,
                "word_2": word_2,
                "corpus": corpus_name,
                "similarity": similarity,
            })

    return pd.DataFrame(rows)

The next cell defines the word pairs we want to compare, runs them through `word_pair_similarity_table()`, and displays the resulting dataframe. You can change the `word_pairs` list to ask different questions. For example, you might compare terms related to gender, family, labor, emotion, place, religion, or any other theme you want to investigate.

In [ ]:
word_pairs = [
    ("money", "marriage"),
    ("money", "work"),
    ("love", "marriage"),
    ("home", "house"),
    ("father", "mother"),
    ("woman", "man"),
    ("child", "mother"),
    ("letter", "write"),
    ("london", "town"),
    ("love", "pain")
]

pair_similarities = word_pair_similarity_table(word_pairs)
pair_similarities

Of course, this data might be easier to assess if we visualize it. Let's prepare the data for visualization then create a visualization with `Seaborn` and `Matplotlib`.

In [ ]:
pair_similarity_plot = pair_similarities.copy()
pair_similarity_plot["pair"] = pair_similarity_plot["word_1"] + " / " + pair_similarity_plot["word_2"]

plt.figure(figsize=(9, 5))
sns.barplot(
    data=pair_similarity_plot,
    x="similarity",
    y="pair",
    hue="corpus",
)
plt.title("Word-Pair Similarities In Two Word2Vec Models")
plt.xlabel("Cosine similarity")
plt.ylabel("Word pair")
plt.tight_layout()


A higher bar means the two words are closer within that corpus's embedding space, meaning the model found them to be more contextually similar. This is one way to study word meaning computationally: instead of asking what a word means in isolation, we ask which other words it behaves like in a specific corpus. If `money / marriage` has a higher score in Austen than in Dickens, that does not prove a single interpretation by itself. It suggests that, in this model, those two words appear in more similar textual environments in Austen's novels than in Dickens's novels. Good interpretation should combine this visual evidence with nearest-neighbor tables and close reading of passages where the words appear.


## Step 6: Visualize Selected Words With PCA

Word vectors have many dimensions. In this notebook, each Word2Vec vector has 100 dimensions, meaning each word is represented by 100 learned numbers. You can think of each dimension as one coordinate in a high-dimensional space. A word is not located on a flat map with only `x` and `y` positions; it is located by 100 coordinates at once. Those dimensions are learned by the model, so they usually do not have simple labels like `class`, `gender`, or `emotion`. Meaning comes from the whole 100-number pattern and from how that pattern relates to other words' patterns.

__PCA__, or __principal component analysis__, lets us project those vectors down to two dimensions for visualization. PCA looks for the directions in the high-dimensional data that account for the most variation, then uses those directions as new axes. Here, PCA will take the 100-dimensional vectors for our selected words and create two new coordinates, `PCA 1` and `PCA 2`, that summarize as much of the variation among those selected words as possible.

This makes plotting possible, but it is a simplification. The plot is not the original embedding space. It is a two-dimensional projection of a much richer 100-dimensional space.

We'll plot the same selected words in the Austen and Dickens models. Each plot should be interpreted inside its own model space.

The function below plots selected word vectors from one model. It first checks which requested words are actually present in the model's vocabulary. Then it collects the vectors for those available words and converts them into a NumPy array. Next, it creates a `PCA` model with `n_components=2`, which means we want two coordinates for each word. `fit_transform()` learns the PCA projection from the selected vectors and returns the two-dimensional coordinates.

The rest of the function uses Matplotlib to draw the plot. It scatters the PCA coordinates, labels each point with its word, and adds a title and axis labels. Because PCA is fit separately each time this function runs, the Austen plot and Dickens plot should each be interpreted inside its own model space.

In [ ]:
def plot_word_vectors(model, words, title, ax):

    available_words = []
    for word in words:
        if word in model.wv:
            available_words.append(word)

    vectors = []
    for word in available_words:
        vectors.append(model.wv[word])

    vectors = np.array(vectors)

    pca = PCA(n_components=2)
    coordinates = pca.fit_transform(vectors)

    ax.scatter(coordinates[:, 0], coordinates[:, 1], color="steelblue")

    for word, x_coord, y_coord in zip(available_words, coordinates[:, 0], coordinates[:, 1]):
        ax.text(x_coord, y_coord, word, fontsize=9)

    ax.set_title(title)
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")

The next cell chooses a set of words to visualize, creates two side-by-side plots, and sends the same word list to the Austen and Dickens models. You can change `selected_words` to focus on a different theme or question. For example, you might build a list around kinship terms, economic terms, place names, emotion words, or words that appeared in surprising nearest-neighbor results.

In [ ]:
selected_words = [
    "money", "marriage", "love", "home", "house", "work",
    "child", "mother", "father", "woman", "man", "letter",
    "write", "london", "town", "death", "hope",
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_word_vectors(austen_model, selected_words, "Austen Word2Vec", axes[0])
plot_word_vectors(dickens_model, selected_words, "Dickens Word2Vec", axes[1])
plt.tight_layout()


These plots are helpful, but they are not maps of actual meaning. PCA compresses 100 dimensions into 2. Distances can shift in that compression.

Use the plots to notice possible patterns, then check those patterns with nearest-neighbor tables and close reading. PCA is useful because it gives us a visual entry point into a space we otherwise cannot see. Clusters can suggest that a group of words occupy similar positions in the model. Isolated words can suggest that a term behaves differently from the rest of the selected set.

But PCA also has important limits. It compresses 100 dimensions into 2, so some distances will be distorted or lost. The plot depends on which words you choose; adding or removing words can change the projection. The axes do not have obvious semantic meanings, and the positions are not directly comparable across separately trained models. Treat the visualization as a tool for generating questions, not as a final map of meaning.

## Step 7: Use spaCy Pretrained Word Vectors

So far, we've trained small Word2Vec models on our own corpora. These give us a sense of word meanings within the context of those corpora. spaCy also has pretrained word vectors, though.

The small English model, `en_core_web_sm`, does not include full word vectors. For this section, we'll load `en_core_web_md`, the medium English model. This is a pretrained English spaCy pipeline. It includes components for tokenization, part-of-speech tagging, dependency parsing, sentence segmentation, named entity recognition, attribute rules, lemmatization, and word vectors.

The vector data in `en_core_web_md` gives many vocabulary items 300-dimensional vectors. These are not trained on our Austen and Dickens texts. The current spaCy model documentation describes the model as trained for English web written text, and its vector source as Explosion Vectors trained from OSCAR, Wikipedia, OpenSubtitles, and WMT News Crawl data. That means these vectors represent broad patterns from a large, mixed, mostly modern textual environment, not the local semantic patterns of nineteenth-century fiction.

This makes `en_core_web_md` useful as a comparison point. If the Austen or Dickens Word2Vec model gives a word pair a different similarity score from spaCy's pretrained vectors, that difference may tell us something about corpus specificity. But we should be careful: the models differ in training data, size, vector dimensions, preprocessing, and training method. We are comparing evidence from different modeling situations, not measuring one single universal meaning.

This may take a minute to download the first time you run it.

In [ ]:
vector_nlp = load_spacy_model("en_core_web_md")

print(vector_nlp("money").has_vector)
print(vector_nlp("money").vector.shape)

spaCy computes similarity between tokens or short documents using their vectors. When we call `vector_nlp("money")`, spaCy returns a short `Doc` object containing one token; when we call `.similarity()` on two such objects, spaCy compares their vector representations.

Let's test a few simple similarities. The `.similarity()` method returns a cosine similarity score, much like the Word2Vec similarity scores above. For a one-word `Doc`, the score is based on that word's vector. For a longer `Doc`, spaCy typically uses the document's vector representation, which is based on the vectors of the tokens in the document.

In [ ]:
print("money / marriage:", vector_nlp("money").similarity(vector_nlp("marriage")))
print("father / mother:", vector_nlp("father").similarity(vector_nlp("mother")))
print("letter / write:", vector_nlp("letter").similarity(vector_nlp("write")))

Now let's compare our corpus-specific Word2Vec similarities with spaCy's pretrained similarities.

Remember: this is not an apples-to-apples comparison. The spaCy model uses vectors from a much larger, mixed training context. Our Word2Vec models were trained on small author corpora. The point is to see how a broad pretrained model differs from models trained on our specific texts.

The function below creates a spaCy version of our word-pair similarity table. It loops through the same list of word pairs, turns each word into a one-word spaCy `Doc`, calculates similarity with `.similarity()`, and stores the result in a dataframe. The `corpus` column is labeled `spaCy pretrained` so we can combine this table with the Austen and Dickens Word2Vec results while still knowing where each score came from.

In [ ]:
def spacy_pair_similarity_table(pairs, nlp):
    rows = []

    for word_1, word_2 in pairs:
        rows.append({
            "word_1": word_1,
            "word_2": word_2,
            "corpus": "spaCy pretrained",
            "similarity": nlp(word_1).similarity(nlp(word_2)),
        })

    return pd.DataFrame(rows)

The next cell calculates spaCy similarities for the same `word_pairs` list and then combines those results with the Austen and Dickens similarity table. The resulting dataframe lets us compare three sources of evidence: Austen Word2Vec, Dickens Word2Vec, and spaCy's pretrained vectors.

In [ ]:
spacy_pair_similarities = spacy_pair_similarity_table(word_pairs, vector_nlp)
all_pair_similarities = pd.concat([
    pair_similarities,
    spacy_pair_similarities,
], ignore_index=True)

all_pair_similarities

The visualization code below prepares the combined dataframe for plotting by creating a readable `pair` label for each word pair. It then uses Seaborn to draw a grouped bar chart, with one bar for each model or corpus. This makes it easier to see when a word pair is especially close in one model but not in another.

In [ ]:
all_pair_plot = all_pair_similarities.copy()
all_pair_plot["pair"] = all_pair_plot["word_1"] + " / " + all_pair_plot["word_2"]

plt.figure(figsize=(10, 6))
sns.barplot(
    data=all_pair_plot,
    x="similarity",
    y="pair",
    hue="corpus",
)
plt.title("Corpus-Specific Word2Vec Similarities Compared With spaCy Vectors")
plt.xlabel("Cosine similarity")
plt.ylabel("Word pair")
plt.tight_layout()

### Reading The Three Models Together

The Austen and Dickens models are small and corpus-specific. They can highlight local patterns in these texts, but they are noisy. By "noisy", I mean that the results may be unstable or overly influenced by the limits of the corpus: a small number of novels, uneven word frequencies, rare words, repeated names, chapter headings, OCR or transcription artifacts, preprocessing choices, and the randomness involved in training. A surprising neighbor may be meaningful, but it may also be an artifact of limited data.

The spaCy model is larger and more stable, but it's not specific to Austen, Dickens, nineteenth-century fiction, or our course corpus. It gives us a broader reference point trained from large mixed sources, including web, Wikipedia, subtitles, and news-crawl data. Because that training context is much broader and less directly tied to the texts we're interpreting, its results can be harder to connect to a specific passage, author, genre, or historical moment. In exchange, the pretrained model is less vulnerable to some of the small-corpus instability we see in our author-specific Word2Vec models. But at the same time, it can be harder to pinpoint exactly what's driving its weights.

A useful DH question might be:

Where do corpus-specific embeddings diverge from a broader pretrained model, and what might that difference reveal about genre, period, author, corpus construction and the semantic relationships between these words?